# 06 - Single-file and batch audio prediction

This notebook loads the model selected using validation macro-F1 in notebook 05. It demonstrates prediction on WAV files without retraining or reading the held-out test set. Confidence scores are model scores and should not be described as calibrated probabilities.

**Run notebooks 01-05 first.**

## 1. Imports and load the selected model

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.prediction import AudioCommandPredictor
from src.preprocessing import load_audio_segment

predictor = AudioCommandPredictor(PROJECT_ROOT)
print('Selected model:', predictor.model_name)
print('Recognised classes:', predictor.labels)


## 2. Single-file prediction

Set `AUDIO_PATH` to any one-second or longer WAV recording. Audio is converted to mono, resampled to 16 kHz, and padded or trimmed to one second.

In [ ]:
AUDIO_PATH = PROJECT_ROOT / 'data' / 'inference_samples' / 'my_command.wav'  # change this path
if AUDIO_PATH.is_file():
    single_result = predictor.predict_one(AUDIO_PATH)
    display(pd.DataFrame(single_result['top_predictions']))
    print('Predicted command:', single_result['predicted_command'])
    print('Confidence score:', round(single_result['confidence_score'], 4))
    print(single_result['confidence_note'])
else:
    print('Add a WAV file at:', AUDIO_PATH)


## 3. Batch prediction

Place several WAV files in the inference folder. The output CSV contains one row per file.

In [ ]:
INFERENCE_DIR = PROJECT_ROOT / 'data' / 'inference_samples'
OUTPUT_CSV = PROJECT_ROOT / 'outputs' / 'batch_predictions.csv'
audio_files = sorted(INFERENCE_DIR.glob('*.wav')) if INFERENCE_DIR.is_dir() else []
if audio_files:
    batch_results = predictor.predict_files(audio_files, output_csv=OUTPUT_CSV)
    display(batch_results)
    print('Saved batch predictions to:', OUTPUT_CSV)
else:
    print('No WAV files found. Add WAV recordings to:', INFERENCE_DIR)


## 4. How to interpret the output

- `predicted_command` is one of the 12 trained labels: ten commands, `unknown`, or `silence`.
- `confidence_score` is the model's score for the selected class. It is useful for ranking but is not calibrated confidence and should not be presented as a guaranteed probability.
- `top_predictions` lists the highest-scoring classes for one recording.
- `batch_predictions.csv` is the bulk output and can be used by the later API and dashboard.

The prediction module does not retrain models. Retraining will be implemented as a separate controlled workflow.